# QLoRA fine-tune + DSPy RAG on SQuAD, end to end

**Goal:** fine-tune Llama-2-7B with QLoRA (4-bit), show the GPU-memory saving, then build a
retrieval-augmented QA pipeline in DSPy whose answerer **is that fine-tuned model**, compile
its prompt with BootstrapFewShot, and report token-level F1 before vs after compiling.

The fine-tuned model is served **in-process** (4-bit base + LoRA adapter wrapped as a
`dspy.BaseLM`), so there is no merge / GGUF / Ollama step and no large scratch-disk peak.
Run on a FRESH GPU kernel, top to bottom. ~20-40 min on a T4.

## 1. Install

In [1]:
# torchao: stale image version breaks newer transformers. wandb: its protobuf module
# breaks trl's import. Remove both before importing transformers/trl. sentencepiece +
# protobuf are needed by the Llama tokenizer.
!pip uninstall -y -q torchao wandb
!pip install -q -U "dspy>=2.6.0" sentence-transformers "datasets>=2.20.0" scikit-learn \
    "transformers>=4.46.0" "peft>=0.13.0" "trl>=0.12.0" "bitsandbytes>=0.43.0" \
    "accelerate>=0.34.0" sentencepiece protobuf
print("installs done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 9.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 97.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.9 MB/s eta 0:00:0000:010:

## 2. GPU check + dtype

In [2]:
import torch
assert torch.cuda.is_available(), "No GPU. Enable a GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))
bf16_ok = torch.cuda.is_bf16_supported()           # T4 -> False (fp16); A100/L4 -> True
compute_dtype = torch.bfloat16 if bf16_ok else torch.float16
print("bf16 supported:", bf16_ok, "| compute dtype:", compute_dtype)

BASE_MODEL  = "NousResearch/Llama-2-7b-hf"          # ungated mirror of Llama-2-7B
ADAPTER_DIR = "llama2-7b-qlora-squad"

GPU: Tesla T4
bf16 supported: True | compute dtype: torch.bfloat16


## 3. Load SQuAD

Real QA benchmark, native metric is token-level F1, maps onto RAG. Sizes are capped so the
local-model eval finishes; raise them for a heavier run.

In [3]:
from datasets import load_dataset

N_TRAIN_SFT = 600    # examples for the QLoRA fine-tune
N_TRAIN_RAG = 50     # labeled examples BootstrapFewShot compiles from
N_EVAL      = 50     # held-out questions scored with F1

ds = load_dataset("rajpurkar/squad")
def to_rows(rows):
    out = []
    for r in rows:
        ans = r["answers"]["text"] or [""]
        out.append({"q": r["question"], "c": r["context"], "a": ans[0], "answers": ans})
    return out

sft_rows  = to_rows(ds["train"].shuffle(seed=0).select(range(N_TRAIN_SFT)))
rag_rows  = to_rows(ds["train"].shuffle(seed=1).select(range(N_TRAIN_RAG)))
eval_rows = to_rows(ds["validation"].shuffle(seed=0).select(range(N_EVAL)))
print(f"sft={len(sft_rows)} rag-train={len(rag_rows)} eval={len(eval_rows)}")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

sft=600 rag-train=50 eval=50


## 4. QLoRA (4-bit) fine-tune of Llama-2-7B

Loads the base in 4-bit NF4 with double quantization, attaches LoRA to all linear layers,
trains only the adapters. `max_steps` caps the demo; remove it for a full run.

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

PROMPT = ("### Instruction:\nAnswer the question using only the context. Reply with the "
          "shortest exact span that answers it.\n\n"
          "### Context:\n{c}\n\n### Question:\n{q}\n\n### Answer:\n{a}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sft_dataset = Dataset.from_list(
    [{"text": PROMPT.format(c=r["c"], q=r["q"], a=r["a"]) + tokenizer.eos_token} for r in sft_rows])

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=compute_dtype,
                                bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config,
                                             device_map="auto")
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

trainer = SFTTrainer(
    model=model,
    args=SFTConfig(output_dir=ADAPTER_DIR, num_train_epochs=1, max_steps=80,
                   per_device_train_batch_size=2, gradient_accumulation_steps=4,
                   gradient_checkpointing=True, learning_rate=2e-4, lr_scheduler_type="cosine",
                   warmup_ratio=0.03, optim="paged_adamw_8bit", bf16=bf16_ok, fp16=not bf16_ok,
                   logging_steps=10, save_strategy="no", max_length=1024,
                   dataset_text_field="text", report_to="none"),
    train_dataset=sft_dataset, peft_config=lora, processing_class=tokenizer)
trainer.train()
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter saved to", ADAPTER_DIR)

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

...

Adding EOS to train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,1.674481
20,1.401242
30,1.408274
40,1.363020
50,1.374908
60,1.312063
70,1.382773
80,1.287259


adapter saved to llama2-7b-qlora-squad


## 5. Memory comparison: 4-bit vs 16-bit

Counts true params on the meta device (zero memory) and measures the 4-bit footprint with
`get_memory_footprint()`. We can't use `sum(p.numel())` on the 4-bit model: bitsandbytes
packs two NF4 weights per byte, so quantized layers report ~half their real size, which
would corrupt the 16-bit estimate.

In [5]:
import gc
from accelerate import init_empty_weights
from transformers import AutoConfig

del trainer, model
gc.collect(); torch.cuda.empty_cache()
GIB = 1024 ** 3

cfg = AutoConfig.from_pretrained(BASE_MODEL)
with init_empty_weights():
    _meta = AutoModelForCausalLM.from_config(cfg)
n_params = sum(p.numel() for p in _meta.parameters())
del _meta; gc.collect()

_m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config,
                                          device_map={"": 0})
fourbit_gib = _m.get_memory_footprint() / GIB
del _m; gc.collect(); torch.cuda.empty_cache()

bf16_gib = (n_params * 2) / GIB   # 2 bytes/param if loaded in fp16/bf16
print(f"Parameters:       {n_params/1e9:.2f} B")
print(f"16-bit footprint: {bf16_gib:.2f} GiB (computed from true param count)")
print(f"4-bit footprint:  {fourbit_gib:.2f} GiB (measured)")
print(f"Memory reduction: {(1 - fourbit_gib/bf16_gib)*100:.0f}%")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Parameters:       6.74 B
16-bit footprint: 12.55 GiB (computed from true param count)
4-bit footprint:  3.50 GiB (measured)
Memory reduction: 72%


## 6. Serve the fine-tuned model in-process

Load the 4-bit base, attach the trained LoRA adapter (no merge), and wrap it as a tiny
`dspy.BaseLM`. `device_map={"": 0}` pins the whole model to GPU 0 so nothing offloads to
CPU (CPU offload silently makes generation minutes-per-call). `LenientChatAdapter` keeps
DSPy from crashing if the model doesn't emit field markers; left truncation keeps the
current question at the end of the prompt once few-shot demos make it long.

In [6]:
import dspy
from types import SimpleNamespace
from peft import PeftModel
from dspy.adapters.chat_adapter import ChatAdapter

class LocalHFLM(dspy.BaseLM):
    """DSPy LM backed by the local fine-tuned model (no API, no Ollama)."""
    def __init__(self, model, tokenizer, max_tokens=64, temperature=0.0):
        super().__init__(model="local/llama2-ft", model_type="chat",
                         temperature=temperature, max_tokens=max_tokens)
        self.hf, self.tok = model, tokenizer
    def forward(self, prompt=None, messages=None, **kwargs):
        merged = {**self.kwargs, **kwargs}          # BaseLM stores these in self.kwargs
        temp = merged.get("temperature", 0.0)
        text = ("\n\n".join(m.get("content", "") for m in messages) + "\n\n"
                if messages else (prompt or ""))
        inputs = self.tok(text, return_tensors="pt", truncation=True,
                          max_length=3072).to(self.hf.device)
        gk = dict(max_new_tokens=merged.get("max_tokens", 64),
                  pad_token_id=self.tok.pad_token_id)
        gk.update(dict(do_sample=True, temperature=temp) if temp and temp > 0 else dict(do_sample=False))
        with torch.no_grad():
            out = self.hf.generate(**inputs, **gk)
        gen = self.tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=gen))],
                               usage={}, model=self.model, _hidden_params={})

class LenientChatAdapter(ChatAdapter):
    """If the model omits DSPy's field markers, fall back to its first non-empty line."""
    def parse(self, signature, completion):
        try:
            return super().parse(signature, completion)
        except Exception:
            fields = list(signature.output_fields)
            first = next((l.strip() for l in completion.splitlines() if l.strip()),
                         completion.strip())
            return {f: (first if f == fields[-1] else "") for f in fields}

tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.truncation_side = "left"

_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config,
                                             device_map={"": 0})
ft_model = PeftModel.from_pretrained(_base, ADAPTER_DIR).eval()
print("model device:", next(ft_model.parameters()).device,
      "| VRAM:", round(torch.cuda.memory_allocated()/GIB, 2), "GiB")

dspy.configure(lm=LocalHFLM(ft_model, tok, max_tokens=64, temperature=0.0),
               adapter=LenientChatAdapter())
print("LM check:", dspy.settings.lm("Context: Paris is the capital of France.\n\n"
                                     "Question: What is the capital?\n\nAnswer:")[0][:60])

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

model device: cuda:0 | VRAM: 4.26 GiB
LM check: Paris


## 7. DSPy RAG pipeline: baseline vs BootstrapFewShot-compiled

Retriever embeds the SQuAD context paragraphs (plain numpy cosine top-k, deepcopy-safe so
compiling is clean). The QA module answers from retrieved context; the metric is SQuAD
token-level F1. We score the uncompiled pipeline, compile the prompt from labeled examples
with BootstrapFewShot, and score again.

In [7]:
import re, string, statistics
from collections import Counter
from sentence_transformers import SentenceTransformer
from dspy.teleprompt import BootstrapFewShot

# retriever over unique SQuAD contexts; embed once on GPU
corpus = list(dict.fromkeys(r["c"] for r in rag_rows + eval_rows))
_st = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
_corpus_emb = _st.encode(corpus, convert_to_numpy=True, normalize_embeddings=True)
def retrieve(question, k=4):
    q = _st.encode([question], convert_to_numpy=True, normalize_embeddings=True)[0]
    return [corpus[i] for i in (_corpus_emb @ q).argsort()[::-1][:k]]

trainset = [dspy.Example(question=r["q"], answer=r["a"], answers=r["answers"]).with_inputs("question") for r in rag_rows]
evalset  = [dspy.Example(question=r["q"], answer=r["a"], answers=r["answers"]).with_inputs("question") for r in eval_rows]

# SQuAD token-level F1
def _norm(t):
    t = (t or "").lower()
    t = "".join(c for c in t if c not in set(string.punctuation))
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", t).split())
def squad_f1(pred, gold):
    p, g = _norm(pred).split(), _norm(gold).split()
    if not p or not g: return float(p == g)
    same = sum((Counter(p) & Counter(g)).values())
    if not same: return 0.0
    pr, rc = same/len(p), same/len(g)
    return 2*pr*rc/(pr+rc)
def best_f1(pred, golds): return max((squad_f1(pred, x) for x in golds), default=0.0)
def boot_metric(ex, pred, trace=None): return best_f1(getattr(pred, "answer", ""), ex.answers) >= 0.6

class QA(dspy.Signature):
    """Answer the question using only the context. Reply with the shortest exact span."""
    context = dspy.InputField()
    question = dspy.InputField()
    answer = dspy.OutputField(desc="shortest exact answer span")

class RAG(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.Predict(QA)
    def forward(self, question):
        return self.generate(context="\n".join(retrieve(question)), question=question)

def evaluate(prog, examples, label=""):
    rows, fails = [], 0
    for i, ex in enumerate(examples, 1):
        try:
            ans = getattr(prog(question=ex.question), "answer", "") or ""
        except Exception:
            ans, fails = "", fails + 1
        rows.append((round(best_f1(ans, ex.answers), 4), ans, ex.answer))
        if i % 10 == 0:
            print(f"  [{label}] {i}/{len(examples)}  F1={statistics.mean(r[0] for r in rows):.3f}")
    return statistics.mean(r[0] for r in rows), rows, fails

base_mean, base_rows, _ = evaluate(RAG(), evalset, "baseline")
print(f"\nBaseline F1 = {base_mean:.3f}  (perfect {sum(r[0]==1.0 for r in base_rows)}/{len(base_rows)})")

compiled = BootstrapFewShot(metric=boot_metric, max_bootstrapped_demos=2,
                            max_labeled_demos=2, max_rounds=1, max_errors=20
                            ).compile(student=RAG(), trainset=trainset)

comp_mean, comp_rows, _ = evaluate(compiled, evalset, "compiled")
print(f"Compiled F1 = {comp_mean:.3f}  (perfect {sum(r[0]==1.0 for r in comp_rows)}/{len(comp_rows)})")
print(f"\nLift from compiling: {comp_mean - base_mean:+.3f} ({base_mean:.3f} -> {comp_mean:.3f})")
print("\nBest compiled examples:")
for f, a, g in sorted(comp_rows, reverse=True)[:5]:
    print(f"  F1={f:.2f} | pred={a[:50]!r} | gold={g[:50]!r}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  [baseline] 10/50  F1=0.614
  [baseline] 20/50  F1=0.599
  [baseline] 30/50  F1=0.510
  [baseline] 40/50  F1=0.509
  [baseline] 50/50  F1=0.535

Baseline F1 = 0.535  (perfect 22/50)


  8%|▊         | 4/50 [00:35<06:48,  8.87s/it]


Bootstrapped 2 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
  [compiled] 10/50  F1=0.872
  [compiled] 20/50  F1=0.886
  [compiled] 30/50  F1=0.824
  [compiled] 40/50  F1=0.805
  [compiled] 50/50  F1=0.843
Compiled F1 = 0.843  (perfect 38/50)

Lift from compiling: +0.309 (0.535 -> 0.843)

Best compiled examples:
  F1=1.00 | pred='thunderstorms' | gold='thunderstorms'
  F1=1.00 | pred='the upper classes' | gold='upper classes'
  F1=1.00 | pred='the greatest common divisor is one' | gold='their greatest common divisor is one'
  F1=1.00 | pred='the founding of new Protestant churches' | gold='the founding of new Protestant churches'
  F1=1.00 | pred='the European Parliament and the Council of the Eur' | gold='the European Parliament and the Council of the Eur'


## Notes

- The answerer is the fine-tuned model (4-bit base + LoRA adapter via `LocalHFLM`), served
  in-process. No GGUF/Ollama, so no large scratch-disk peak.
- Step 5 memory result is data-independent: pure 4-bit vs 16-bit weight storage.
- Expected numbers on T4: ~6.7B params, 16-bit ~12.5 GiB vs 4-bit ~3.9 GiB (~69% saving);
  F1 ~0.42 baseline -> ~0.81 compiled. Much of the F1 lift is the demos teaching the output
  format, which is exactly what BootstrapFewShot optimizes.
- Speed knobs: lower `N_EVAL`, lower `max_steps` in step 4, lower `k` in `retrieve`.
- For a higher absolute F1, fine-tune Llama-2-7b-**chat** instead of the base model; it
  follows the field format more reliably.